## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.utils import resample
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Parameter & Datasets

In [3]:
current_date = datetime.now()

start_date = '20240512'
end_date = '20240519'
print(start_date, ' to ' ,end_date)

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')

segment_date = end_dt.strftime('%Y-%m-%d')

20240512  to  20240519


### Query 1

In [4]:
# Initialize an empty list to hold the DataFrames
df_data = pd.DataFrame()

# Loop through each date in the range
current_dt = start_dt
while current_dt <= end_dt:
    # Format the current date as a string in 'YYYYMMDD' format
    current_date_str = current_dt.strftime('%Y%m%d')

    data_query = f"""

        with olf_data as (

            select
                yyyymmdd,
                city_name,
                service_obj_service_name service_name,
                customer_id,
                order_id,
                order_status,
                modified_order_status,
                spd_fraud_flag,
                CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_to_cancel_in_sec
            from
                orders.order_logs_fact
            where 
                yyyymmdd = '{current_date_str}'
                and city_name in ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata')
                and service_obj_service_name in ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')
            ),

            orderLogsFact as (

            select
                yyyymmdd,
                customer_id,
                city_name,
                service_name
            from
                olf_data
            group by 
                1,2,3,4
            ),

            intervention as (

            SELECT 
                yyyymmdd,
                userId,
                group_tc,
                max(coalesce(dispatch_ad_request, dispatch_ad_response, dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v1,
                max(coalesce(dispatch_ad_response, dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v2,
                max(coalesce(dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v3
            FROM
                (
                SELECT 
                    yyyymmdd,
                    userId,
                    'test' group_tc,
                    CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_request,
                    CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_response,
                    CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_viewable_impression,
                    CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_click
                FROM 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                WHERE  
                    yyyymmdd = '{current_date_str}'
                    and responseType = 'GAMBanner'
                    and pagename = 'CaptainSearchScreen'
                    and city in ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata')
                    and service in ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')                
                    and userId IN (select distinct customer_id from orderLogsFact)

                )
            GROUP BY 1,2,3
            ),

            customer_segment as (

            select 
                customer_id,
                taxi_lifetime_stage customer_segment

            from
                datasets.iallocator_customer_segments
            where 
                run_date = '{segment_date}'
                and taxi_recency_segment != 'INACTIVE'
            ),

            experiment_base as (

            select
                a.yyyymmdd,
                a.city_name,
                a.service_name,
                a.customer_id,
                coalesce(b.group_tc, 'control') group_tc,
                c.customer_segment
            from
                orderLogsFact a
            left join 
                intervention b
                on a.yyyymmdd = b.yyyymmdd
                and a.customer_id = b.userId
            left join 
                customer_segment c
                on a.customer_id = c.customer_id

            group by 1,2,3,4,5,6

            )


            select
                a.*,

                count(distinct order_id) gross_orders,
                count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
                count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm,
                count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra,
                count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara,
                count(distinct case when modified_order_status = 'aborted' then order_id end) aborted,
                count(distinct case when modified_order_status = 'expired' then order_id end) expired,

                avg(CASE WHEN modified_order_status = 'COBRM' THEN time_to_cancel_in_sec END) as avg_cobrm_time,
                avg(CASE WHEN modified_order_status = 'COBRA' THEN time_to_cancel_in_sec END) as avg_cobra_time,
                avg(CASE WHEN modified_order_status = 'OCARA' THEN time_to_cancel_in_sec END) as avg_ocara_time

            from 
                experiment_base a
            join 
                olf_data b
                on a.yyyymmdd = b.yyyymmdd
                and a.customer_id = b.customer_id
                and a.city_name = b.city_name
            group by 1,2,3,4,5,6

    """

    # Execute the query and get the result as a DataFrame
    df_day = pd.read_sql(data_query, connection)
    
    # Concatenate the result with the existing DataFrame
    df_data = pd.concat([df_data, df_day], ignore_index=True)
    
    # Move to the next date
    current_dt += timedelta(days=1)

# Display the concatenated DataFrame
df_data

,yyyymmdd,city_name,service_name,customer_id,group_tc,customer_segment,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time
0,20240512,Hyderabad,Auto,626949e640b3e5c5c9c0a868,control,COMMITTED,1,1,0,0,0,0,0,NaN,NaN,NaN
1,20240512,Delhi,CabEconomy,632bf5d4868f899277b13372,test,COMMITTED,5,1,0,3,1,0,0,NaN,60.0,268.0
2,20240512,Pune,Auto,64002e696ee30010063824ce,test,COMMITTED,1,1,0,0,0,0,0,NaN,NaN,NaN
3,20240512,Bangalore,Auto,5b120e43812a634e420f15fa,control,HOOK,1,0,0,0,1,0,0,NaN,NaN,304.0
4,20240512,Delhi,Link,632341845d3af40e86021dc1,control,COMMITTED,2,1,0,0,0,0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10495928,20240519,Bangalore,Auto,5d6f1815a24c96105e370a5d,control,COMMITTED,4,2,0,1,0,0,1,NaN,192.0,NaN
10495929,20240519,Hyderabad,Auto,62fa2006a42e8131dd6ed8d6,control,HOOK,3,2,0,0,1,0,0,NaN,NaN,265.0
10495930,20240519,Chennai,Auto,65dc5f4f8f892e4ff9a74dc0,control,COMMITTED,1,0,0,0,1,0,0,NaN,NaN,341.0
10495931,20240519,Bangalore,Link,646614766f8f1bad18b8460f,test,COMMITTED,2,2,0,0,0,0,0,NaN,NaN,NaN


### Query 2

In [5]:
# Initialize an empty list to hold the DataFrames
df_telemetry_ads = pd.DataFrame()

# Loop through each date in the range
current_dt = start_dt
while current_dt <= end_dt:
    # Format the current date as a string in 'YYYYMMDD' format
    current_date_str = current_dt.strftime('%Y%m%d')

    telemetry_ads = f"""

    with olf_data as (

            select
                yyyymmdd yyyymd,
                customer_id,
                city_name,
                service_obj_service_name service_name
            from
                orders.order_logs_fact
            where 
                yyyymmdd = '{current_date_str}'
                and city_name in ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata')
                and service_obj_service_name in ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')
            group by 1,2,3,4
            )

        SELECT 
            yyyymmdd,
            city_name,
            service_name,
            userId,

            -- pagename
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS css_request,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS css_response,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS css_viewable_impression,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS css_click,

            TRY(COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS pos_request,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS pos_response,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS pos_viewable_impression,
            TRY(COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS pos_click,

            --screen_slot_name
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS onTheWay_request,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS onTheWay_response,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS onTheWay_viewable_impression,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS onTheWay_click,    

            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS arrived_request,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS arrived_response,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS arrived_viewable_impression,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS arrived_click,    

            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS started_request,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS started_response,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS started_viewable_impression,
            TRY(COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END )) AS started_click

        FROM
        (
        SELECT 
            yyyymmdd,
            city_name,
            service_name,
            userId,
            pagename,
            CASE 
            WHEN pagename IN ('HomeScreen', 'CaptainSearchScreen') THEN pagename
            WHEN lower(slotname) LIKE '%ontheway%' THEN 'onTheWay'
            WHEN lower(slotname) LIKE '%arrived%' THEN 'arrived'
            WHEN lower(slotname) LIKE '%started%' THEN 'started'
            WHEN ltrim(orderstatus) IS NOT NULL THEN orderstatus
            END screen_slot_name,

            eventName,
            hhmmss
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable 
        JOIN
            olf_data
            on yyyymmdd = yyyymd 
            and customer_id = userId
            and city_name = city
            and service_name = service

        WHERE  
            yyyymmdd = '{current_date_str}'
            and responseType = 'GAMBanner'
            and city in ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata')
            and service in ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')        
        )
        GROUP BY 1,2,3,4

    """
    
    # Execute the query and get the result as a DataFrame
    df1_day = pd.read_sql(telemetry_ads, connection)
    
    # Concatenate the result with the existing DataFrame
    df_telemetry_ads = pd.concat([df_telemetry_ads, df1_day], ignore_index=True)
    
    # Move to the next date
    current_dt += timedelta(days=1)

# Display the concatenated DataFrame
df_telemetry_ads

,yyyymmdd,city_name,service_name,userId,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,20240512,Kolkata,Link,638ca7a912ae9f6e1b3aa3c0,0,0,0,0,8,7,4,0,3,5,2,0,2,1,1,0,3,1,1,0
1,20240512,Chennai,CabEconomy,65fb05967af51440f43f13d5,0,0,0,0,12,16,5,0,3,10,2,0,3,2,1,0,6,4,2,0
2,20240512,Delhi,Link,639375d5fc444c74ff14d008,0,0,0,0,8,5,1,0,4,3,0,0,4,2,1,0,0,0,0,0
3,20240512,Chennai,Link,6138e8538709d27434978945,0,0,0,0,5,4,3,0,3,3,2,0,2,1,1,0,0,0,0,0
4,20240512,Kolkata,Link,60ebabe7ac6ff4bb6761e62e,0,0,0,0,4,1,3,0,2,1,3,0,2,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5122293,20240519,Delhi,Link,635aa25cc74d18c7359a9994,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
5122294,20240519,Delhi,Auto,6255368d91957f34da91150f,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5122295,20240519,Bangalore,Bike Lite,63199d7774328c4c3d92708b,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5122296,20240519,Bangalore,Link,633fb0b98521272ac1b13958,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## User Define Function

In [6]:
df_data.head(2)

,yyyymmdd,city_name,service_name,customer_id,group_tc,customer_segment,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time
0,20240512,Hyderabad,Auto,626949e640b3e5c5c9c0a868,control,COMMITTED,1,1,0,0,0,0,0,NaN,NaN,NaN
1,20240512,Delhi,CabEconomy,632bf5d4868f899277b13372,test,COMMITTED,5,1,0,3,1,0,0,NaN,60.0,268.0


In [7]:
def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

In [8]:
customer_agg_dict = {
        'customers': ('customer_id', 'nunique'),
        'gross_orders': ('gross_orders', 'sum'),
        'net_orders': ('net_orders', 'sum'),
        'cobrm': ('cobrm', 'sum'),
        'cobra': ('cobra', 'sum'),
        'ocara': ('ocara', 'sum'),
        'aborted': ('aborted', 'sum'),
        'expired': ('expired', 'sum'),
        'cobrm_time': ('avg_cobrm_time', 'median'),
        'cobra_time': ('avg_cobra_time', 'median'),
        'ocara_time': ('avg_ocara_time', 'median')
}

telemetri_agg_dict = {
        'customers': ('customer_id', 'nunique'),
        'gross_orders': ('gross_orders', 'sum'),
        'net_orders': ('net_orders', 'sum'),
        'cobrm': ('cobrm', 'sum'),
        'cobra': ('cobra', 'sum'),
        'ocara': ('ocara', 'sum'),
        'aborted': ('aborted', 'sum'),
        'expired': ('expired', 'sum'),
        'cobrm_time': ('avg_cobrm_time', 'median'),
        'cobra_time': ('avg_cobra_time', 'median'),
        'ocara_time': ('avg_ocara_time', 'median'),

        'css_request': ('css_request', 'sum'),
        'css_response': ('css_response', 'sum'),
        'css_viewable_impression': ('css_viewable_impression', 'sum'),
        'css_click': ('css_click', 'sum'),

        'pos_request': ('pos_request', 'sum'),
        'pos_response': ('pos_response', 'sum'),
        'pos_viewable_impression': ('pos_viewable_impression', 'sum'),
        'pos_click': ('pos_click', 'sum'),

        'onTheWay_request': ('onTheWay_request', 'sum'), 
        'onTheWay_response': ('onTheWay_response', 'sum'), 
        'onTheWay_viewable_impression': ('onTheWay_viewable_impression', 'sum'),
        'onTheWay_click': ('onTheWay_click', 'sum'), 

        'arrived_request': ('arrived_request', 'sum'), 
        'arrived_response': ('arrived_response', 'sum'),
        'arrived_viewable_impression': ('arrived_viewable_impression', 'sum'), 
        'arrived_click': ('arrived_click', 'sum'), 

        'started_request': ('started_request', 'sum'),
        'started_response': ('started_response', 'sum'), 
        'started_viewable_impression': ('started_viewable_impression', 'sum'), 
        'started_click': ('started_click', 'sum'),
} 

In [9]:
def sampling(df):
    
    df['group_tc'] = df['group_tc'].astype('category')

    # Create an empty DataFrame to store the sampled data
    sampled_df = pd.DataFrame()

    # Group by 'city_name' and 'service_name'
    grouped = df.groupby(['city_name', 'service_name'])

    for (city, service), group in grouped:
        # Get the proportions of 'customer_segment' within the group
        segment_proportions = group['customer_segment'].value_counts(normalize=True)

        # Get the test-control split proportions within the group
        test_control_split = group['group_tc'].value_counts(normalize=True)

        # Stratify by 'customer_segment' to ensure similar proportions in the sample
        stratified_sample = group.groupby('customer_segment', group_keys=False).apply(
            lambda x: x.sample(frac=1)
        )

        # Split the sampled data into test and control groups maintaining the original proportions
        test_group, control_group = train_test_split(
            stratified_sample, test_size=test_control_split['control'], stratify=stratified_sample['group_tc']
        )

        # Concatenate the test and control groups
        sampled_group = pd.concat([test_group, control_group])

        # Append the sampled group to the sampled_df
        sampled_df = pd.concat([sampled_df, sampled_group])

    # Reset index
    sampled_df.reset_index(drop=True, inplace=True)
    
    return sampled_df

## Check biasness

In [10]:
df_data_agg = df_data\
.groupby(['city_name', 
          'service_name',
          'group_tc'])\
.agg(**customer_agg_dict)\
.reset_index()

In [11]:
df_data_agg

,city_name,service_name,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Bangalore,Auto,control,475490,1788402,713101,14092,475630,269054,1525,311693,67.000000,131.000000,280.000000
1,Bangalore,Auto,test,435344,1722327,673693,18741,506361,239298,1208,279883,48.000000,133.000000,289.000000
2,Bangalore,Auto Pool,control,17454,85797,16573,11106,26230,8589,102,23080,81.000000,136.000000,295.000000
3,Bangalore,Auto Pool,test,7050,45352,7372,5441,17494,4127,28,10835,75.000000,138.000000,313.500000
4,Bangalore,Bike Lite,control,7706,26972,9336,204,10611,3591,32,3076,70.000000,131.000000,367.000000
5,Bangalore,Bike Lite,test,7481,29339,9949,455,11823,3793,34,3149,86.500000,134.000000,363.000000
6,Bangalore,CabEconomy,control,199502,593044,219704,5239,174859,106527,681,85421,101.000000,140.000000,322.000000
7,Bangalore,CabEconomy,test,64041,319284,82060,4878,134397,46697,182,50724,79.000000,139.500000,314.500000
8,Bangalore,Link,control,472690,1552535,675417,16666,440247,221472,2456,188978,87.000000,130.000000,343.000000
9,Bangalore,Link,test,100670,530417,159386,9375,215000,70740,438,74169,73.000000,129.000000,305.000000


In [12]:
df_data_agg['g2n'] = calculate_percentage(df_data_agg, 'net_orders', 'gross_orders')
df_data_agg['% cobrm'] = calculate_percentage(df_data_agg, 'cobrm', 'gross_orders')
df_data_agg['% cobra'] = calculate_percentage(df_data_agg, 'cobra', 'gross_orders')
df_data_agg['% ocara'] = calculate_percentage(df_data_agg, 'ocara', 'gross_orders')

df_data_agg[['city_name', 'group_tc','customers', 'g2n', '% cobrm', '% cobra', '% ocara']]

,city_name,group_tc,customers,g2n,% cobrm,% cobra,% ocara
0,Bangalore,control,475490,39.87,0.79,26.60,15.04
1,Bangalore,test,435344,39.12,1.09,29.40,13.89
2,Bangalore,control,17454,19.32,12.94,30.57,10.01
3,Bangalore,test,7050,16.26,12.00,38.57,9.10
4,Bangalore,control,7706,34.61,0.76,39.34,13.31
5,Bangalore,test,7481,33.91,1.55,40.30,12.93
6,Bangalore,control,199502,37.05,0.88,29.48,17.96
7,Bangalore,test,64041,25.70,1.53,42.09,14.63
8,Bangalore,control,472690,43.50,1.07,28.36,14.27
9,Bangalore,test,100670,30.05,1.77,40.53,13.34


In [13]:
df_data_agg[['city_name', 'service_name', 'group_tc','customers']]

,city_name,service_name,group_tc,customers
0,Bangalore,Auto,control,475490
1,Bangalore,Auto,test,435344
2,Bangalore,Auto Pool,control,17454
3,Bangalore,Auto Pool,test,7050
4,Bangalore,Bike Lite,control,7706
5,Bangalore,Bike Lite,test,7481
6,Bangalore,CabEconomy,control,199502
7,Bangalore,CabEconomy,test,64041
8,Bangalore,Link,control,472690
9,Bangalore,Link,test,100670


In [14]:
df_data.query("group_tc == 'test'")["customer_segment"].value_counts(True)

customer_segment
COMMITTED      0.427750
HOOK           0.281174
SUSTENANCE     0.118078
HANDHOLDING    0.099084
DORMANT        0.038725
CHURN_OTB      0.018146
UNKNOWN        0.013974
SOFT_CHURN     0.001624
DETOX          0.001446
Name: proportion, dtype: float64

In [15]:
df_data.query("group_tc == 'control'")["customer_segment"].value_counts(True)

customer_segment
COMMITTED      0.404108
HOOK           0.279932
SUSTENANCE     0.120079
HANDHOLDING    0.106647
DORMANT        0.045096
CHURN_OTB      0.019936
UNKNOWN        0.018022
DETOX          0.003814
SOFT_CHURN     0.002366
Name: proportion, dtype: float64

### Sampling

In [16]:
df_sampled = sampling(df_data)

In [17]:
df_sampled.query("group_tc == 'test'")["customer_segment"].value_counts(True)

customer_segment
COMMITTED      0.427750
HOOK           0.281174
SUSTENANCE     0.118078
HANDHOLDING    0.099084
DORMANT        0.038725
CHURN_OTB      0.018146
UNKNOWN        0.013974
SOFT_CHURN     0.001624
DETOX          0.001446
Name: proportion, dtype: float64

In [18]:
df_sampled.query("group_tc == 'control'")["customer_segment"].value_counts(True)

customer_segment
COMMITTED      0.404108
HOOK           0.279932
SUSTENANCE     0.120079
HANDHOLDING    0.106647
DORMANT        0.045096
CHURN_OTB      0.019936
UNKNOWN        0.018022
DETOX          0.003814
SOFT_CHURN     0.002366
Name: proportion, dtype: float64

In [19]:
df_test = df_sampled\
.groupby(['city_name', 
          'service_name',
#           'customer_segment',
          'group_tc'])\
.agg(**customer_agg_dict)\
.reset_index()

In [20]:
df_test['g2n'] = calculate_percentage(df_test, 'net_orders', 'gross_orders')
df_test['% cobrm'] = calculate_percentage(df_test, 'cobrm', 'gross_orders')
df_test['% cobra'] = calculate_percentage(df_test, 'cobra', 'gross_orders')
df_test['% ocara'] = calculate_percentage(df_test, 'ocara', 'gross_orders')

df_test[['city_name', 
         'service_name', 
         'group_tc','customers', 'g2n', '% cobrm', '% cobra', '% ocara']]

,city_name,service_name,group_tc,customers,g2n,% cobrm,% cobra,% ocara
0,Bangalore,Auto,control,458934,40.57,0.78,26.33,14.89
1,Bangalore,Auto,test,423102,39.71,1.08,29.19,13.78
2,Bangalore,Auto Pool,control,16921,19.64,12.83,30.53,10.01
3,Bangalore,Auto Pool,test,6930,16.45,11.93,38.60,9.08
4,Bangalore,Bike Lite,control,7431,35.38,0.75,39.07,13.06
5,Bangalore,Bike Lite,test,7247,34.46,1.56,40.08,12.72
6,Bangalore,Bike Metro,control,0,NaN,NaN,NaN,NaN
7,Bangalore,Bike Metro,test,0,NaN,NaN,NaN,NaN
8,Bangalore,Bike Pink,control,0,NaN,NaN,NaN,NaN
9,Bangalore,Bike Pink,test,0,NaN,NaN,NaN,NaN


## Merge

In [21]:
df_telemetry_ads.head(1)

,yyyymmdd,city_name,service_name,userId,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,20240512,Kolkata,Link,638ca7a912ae9f6e1b3aa3c0,0,0,0,0,8,7,4,0,3,5,2,0,2,1,1,0,3,1,1,0


In [22]:
df_merge = pd.merge(df_sampled,
                    df_telemetry_ads,
                    how='left',
                    left_on=['yyyymmdd', 'city_name', 'service_name', 'customer_id'],
                    right_on=['yyyymmdd', 'city_name', 'service_name', 'userId'],
                   )
df_merge

,yyyymmdd,city_name,service_name,customer_id,group_tc,customer_segment,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time,userId,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,20240519,Bangalore,Auto,5e0046189a6d72727b1bc9b3,control,COMMITTED,2,2,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20240512,Bangalore,Auto,61346bfcb2fb0dab096a0fcc,control,HOOK,1,1,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20240516,Bangalore,Auto,602e0c7f85e2d8d86fa8aa42,control,SUSTENANCE,1,1,0,0,0,0,0,NaN,NaN,NaN,602e0c7f85e2d8d86fa8aa42,0.0,0.0,0.0,0.0,2.0,7.0,1.0,0.0,2.0,7.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,20240514,Bangalore,Auto,66436819fe041fefaf3352bf,control,UNKNOWN,2,0,0,2,0,0,0,NaN,140.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20240512,Bangalore,Auto,660294d6697bdd0e9139ea68,control,HOOK,3,1,0,2,0,0,0,NaN,81.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10297220,20240519,Pune,CabEconomy,65e91d8a777f4af7c053fd1d,control,HOOK,2,1,0,1,0,0,0,NaN,216.0,NaN,65e91d8a777f4af7c053fd1d,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10297221,20240519,Pune,CabEconomy,6263e39ee98a6492b9949ab0,control,HOOK,4,2,0,1,1,0,0,NaN,153.0,794.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10297222,20240513,Pune,CabEconomy,64abea13de83ef791b1cc285,control,HOOK,2,2,0,0,0,0,0,NaN,NaN,NaN,64abea13de83ef791b1cc285,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0
10297223,20240514,Pune,CabEconomy,638c8120510c06261256c144,control,HOOK,2,1,0,1,0,0,0,NaN,183.0,NaN,638c8120510c06261256c144,0.0,0.0,0.0,0.0,10.0,4.0,4.0,0.0,3.0,1.0,1.0,0.0,2.0,2.0,2.0,0.0,5.0,1.0,1.0,0.0


## Analysis

In [23]:
df_analysis1 =  df_merge\
.groupby(['group_tc'])\
.agg(**telemetri_agg_dict)\
.sort_values(by=['group_tc'])\
.reset_index()

df_analysis1

,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,3827634,13377738,7031955,159006,2638363,2220827,18794,1261563,97.0,130.0,330.0,0.0,0.0,0.0,0.0,14238008.0,5727658.0,3139853.0,14232.0,6541343.0,2994635.0,1410823.0,7226.0,3022588.0,1330373.0,793866.0,1610.0,4684086.0,1402658.0,935167.0,5396.0
1,test,1725329,8016759,3775785,130314,2009952,1262326,6647,810765,95.0,135.0,320.0,9463860.0,1215643.0,813401.0,11519.0,22459461.0,8760719.0,4821155.0,21799.0,10499270.0,4545500.0,2040262.0,10288.0,4543297.0,1977649.0,1140039.0,2042.0,7432483.0,2237577.0,1640857.0,9469.0


In [24]:
df_analysis2 =  df_merge\
.groupby(['group_tc','yyyymmdd'])\
.agg(**telemetri_agg_dict)\
.sort_values(by=['yyyymmdd', 'group_tc'])\
.reset_index()

df_analysis2

,group_tc,yyyymmdd,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,20240512,758761,1606719,841933,17316,311389,287422,2602,140286,105.000000,128.000000,332.0,0.0,0.0,0.0,0.0,1314425.0,497372.0,265456.0,1089.0,596553.0,275621.0,120903.0,623.0,277805.0,75909.0,48730.0,118.0,441007.0,145842.0,95823.0,348.0
1,test,20240512,252197,710121,339323,10120,173791,118113,679,66289,104.000000,133.000000,314.5,871155.0,111882.0,75262.0,594.0,2051050.0,735152.0,399094.0,1443.0,943751.0,396254.0,168543.0,758.0,418375.0,111195.0,66965.0,144.0,690268.0,227703.0,163586.0,541.0
2,control,20240513,805729,1758727,883019,23542,376963,283216,2364,183856,100.000000,132.000000,328.0,0.0,0.0,0.0,0.0,1490769.0,538783.0,285909.0,1507.0,685419.0,304438.0,141544.0,879.0,316356.0,99199.0,60465.0,174.0,490002.0,135146.0,83900.0,454.0
3,test,20240513,288467,861781,387799,16046,233550,128994,702,92625,97.000000,137.000000,316.5,1016236.0,112651.0,75747.0,719.0,2274055.0,752658.0,412489.0,1922.0,1053947.0,417305.0,192283.0,1047.0,471060.0,139484.0,81871.0,164.0,750607.0,195869.0,138336.0,711.0
4,control,20240514,864227,1886681,960753,24609,410714,287922,2276,194152,92.750000,133.000000,326.0,0.0,0.0,0.0,0.0,1808175.0,756342.0,485327.0,2767.0,825817.0,387907.0,224433.0,1503.0,398446.0,156909.0,112626.0,258.0,585167.0,211529.0,148268.0,1006.0
5,test,20240514,318107,969121,444198,17330,263573,140894,680,100064,93.000000,137.000000,313.0,1111195.0,121029.0,81230.0,617.0,2560870.0,1070462.0,695796.0,3943.0,1183670.0,540328.0,300121.0,1960.0,540755.0,210204.0,149500.0,335.0,838231.0,319930.0,246175.0,1648.0
6,control,20240515,859815,1796788,970229,23143,342813,290456,2344,161295,94.000000,130.000000,327.0,0.0,0.0,0.0,0.0,2020526.0,672727.0,374685.0,1059.0,921142.0,345093.0,169377.0,547.0,435627.0,175129.0,106015.0,161.0,665069.0,152505.0,99293.0,351.0
7,test,20240515,336468,980869,478845,16726,235089,152529,738,94328,92.416667,133.666667,317.0,1143386.0,157660.0,106264.0,1275.0,2805840.0,907964.0,501551.0,1326.0,1294223.0,461019.0,213467.0,663.0,575668.0,231296.0,134403.0,144.0,937794.0,215649.0,153681.0,519.0
8,control,20240516,834621,1877495,900138,26182,421483,292575,2628,228215,90.000000,136.500000,338.0,0.0,0.0,0.0,0.0,1933686.0,881374.0,444874.0,1780.0,911910.0,460556.0,186674.0,657.0,398950.0,194398.0,112609.0,185.0,624251.0,226423.0,145592.0,938.0
9,test,20240516,380560,1240779,518341,22252,360550,179908,940,155718,84.000000,143.000000,330.0,1344265.0,189089.0,124641.0,2743.0,3068232.0,1341241.0,678406.0,2835.0,1470790.0,694683.0,265951.0,1009.0,602592.0,286990.0,158521.0,265.0,997102.0,359572.0,253935.0,1561.0


In [25]:
df_analysis3 =  df_merge\
.groupby(['group_tc','city_name'])\
.agg(**telemetri_agg_dict)\
.sort_values(by=['city_name', 'group_tc'])\
.reset_index()

df_analysis3

,group_tc,city_name,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,Bangalore,884056,3978908,1634130,46107,1098582,591132,4560,593538,81.000000,132.000000,308.000000,0.0,0.0,0.0,0.0,2451223.0,999654.0,550596.0,2645.0,1143419.0,515436.0,238237.0,1193.0,483490.0,207826.0,119743.0,201.0,826185.0,276394.0,192618.0,1251.0
1,test,Bangalore,437712,2610237,932460,38153,868193,357109,1845,407618,63.000000,132.600000,295.500000,2767100.0,380311.0,235017.0,3738.0,5601671.0,2269068.0,1236913.0,5365.0,2584106.0,1116037.0,498825.0,2424.0,1121429.0,501901.0,273714.0,423.0,1900332.0,651133.0,464375.0,2518.0
2,control,Chennai,566969,1827287,1147141,25912,238685,305268,2829,99139,106.000000,120.500000,310.000000,0.0,0.0,0.0,0.0,1862576.0,710130.0,364292.0,1832.0,875770.0,362830.0,177130.0,1091.0,376426.0,152283.0,87236.0,169.0,611545.0,195019.0,99926.0,572.0
3,test,Chennai,283561,1103962,664992,19986,163399,191768,1137,59522,112.000000,124.500000,310.000000,1465291.0,198389.0,137201.0,1598.0,3913751.0,1440498.0,815145.0,4425.0,1792654.0,688983.0,326462.0,1937.0,777117.0,323136.0,177881.0,334.0,1346375.0,428380.0,310802.0,2154.0
4,control,Delhi,842038,2901173,1654176,37561,427850,577040,4405,190070,118.000000,125.000000,357.000000,0.0,0.0,0.0,0.0,3591363.0,1448556.0,834272.0,3812.0,1671487.0,753833.0,369845.0,1899.0,747227.0,344019.0,212612.0,461.0,1175102.0,350705.0,251815.0,1452.0
5,test,Delhi,334656,1688777,869897,34060,345599,311066,1374,120368,119.708333,132.500000,348.000000,1921462.0,223809.0,161018.0,1567.0,4572328.0,1699925.0,1004249.0,4107.0,2203330.0,861728.0,439481.0,2020.0,971645.0,431720.0,264136.0,410.0,1400794.0,406477.0,300633.0,1677.0
6,control,Hyderabad,1090237,3280430,1830286,41150,621726,481633,4974,286289,89.000000,135.000000,333.000000,0.0,0.0,0.0,0.0,4343133.0,1695931.0,974278.0,4106.0,1842258.0,839108.0,410658.0,2031.0,1026058.0,454953.0,271903.0,591.0,1478118.0,401872.0,291718.0,1484.0
7,test,Hyderabad,524191,2011228,1016439,31586,478483,297251,1842,180556,84.000000,148.166667,329.000000,2513069.0,326228.0,218888.0,3944.0,6227727.0,2363470.0,1322522.0,6100.0,2787499.0,1230101.0,568087.0,3087.0,1278171.0,559086.0,326986.0,727.0,2166376.0,574286.0,427450.0,2286.0
8,control,Jaipur,151845,468123,239689,3179,108151,84162,680,30899,127.000000,134.333333,383.000000,0.0,0.0,0.0,0.0,514162.0,266279.0,104038.0,454.0,291821.0,196087.0,54026.0,187.0,97443.0,37596.0,23818.0,67.0,125268.0,32597.0,26194.0,200.0
9,test,Jaipur,78493,339979,150618,3825,101213,60208,297,22783,133.000000,135.000000,369.583333,401274.0,35022.0,24965.0,301.0,1196039.0,633698.0,231276.0,1014.0,712235.0,484000.0,124567.0,528.0,201361.0,76358.0,47762.0,85.0,283061.0,73340.0,58947.0,401.0


In [26]:
df_analysis4 =  df_merge\
.groupby(['group_tc','service_name'])\
.agg(**telemetri_agg_dict)\
.sort_values(by=['service_name', 'group_tc'])\
.reset_index()

df_analysis4

,group_tc,service_name,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,Auto,1429873,4484034,2227325,44245,929867,736343,4343,532494,93.5,132.000000,293.000000,0.0,0.0,0.0,0.0,188046.0,58349.0,39184.0,188.0,44125.0,16960.0,8725.0,43.0,32747.0,12385.0,8514.0,26.0,111331.0,29004.0,21945.0,119.0
1,test,Auto,1508301,4905614,2468125,57923,1088872,759654,4052,516148,91.0,137.000000,310.000000,8007573.0,1054056.0,695541.0,10140.0,16800761.0,6530870.0,3619890.0,16450.0,7663948.0,3319302.0,1485205.0,7440.0,3362702.0,1454548.0,827117.0,1505.0,5785605.0,1757026.0,1307570.0,7505.0
2,control,Auto Pool,35407,152247,29666,22829,45338,14285,167,39742,83.0,140.000000,310.000000,0.0,0.0,0.0,0.0,18607.0,8057.0,4483.0,18.0,9506.0,4805.0,2291.0,9.0,3021.0,1364.0,861.0,0.0,6098.0,1888.0,1331.0,9.0
3,test,Auto Pool,15946,91758,17501,12353,33438,8323,43,19977,80.0,144.500000,328.500000,435.0,81.0,53.0,2.0,10058.0,4066.0,2297.0,15.0,6148.0,2722.0,1388.0,11.0,1258.0,551.0,319.0,1.0,2656.0,793.0,590.0,3.0
4,control,Bike Lite,185349,593224,338301,4856,110819,101064,692,34182,106.0,125.666667,363.000000,0.0,0.0,0.0,0.0,124090.0,41759.0,26475.0,125.0,51918.0,18924.0,10310.0,45.0,28564.0,11701.0,7413.0,19.0,43694.0,11134.0,8752.0,61.0
5,test,Bike Lite,185394,707928,400002,6978,142887,118100,673,35429,113.0,132.000000,376.000000,1047744.0,110362.0,81851.0,753.0,2179702.0,840887.0,465143.0,1915.0,1130554.0,478684.0,218937.0,959.0,476839.0,201215.0,126880.0,206.0,573944.0,160988.0,119327.0,750.0
6,control,Bike Metro,24234,66583,40006,302,11733,11657,48,2524,114.0,111.000000,313.166667,0.0,0.0,0.0,0.0,2765.0,883.0,603.0,4.0,567.0,154.0,109.0,0.0,599.0,256.0,179.0,0.0,1599.0,473.0,315.0,4.0
7,test,Bike Metro,25587,76699,45916,545,14226,13178,54,2434,100.0,121.000000,332.000000,84603.0,9917.0,7436.0,64.0,175375.0,65422.0,40253.0,100.0,80466.0,29913.0,16821.0,38.0,46016.0,20752.0,12929.0,10.0,49088.0,14757.0,10503.0,52.0
8,control,Bike Pink,6186,22713,6400,5262,4697,2713,15,3578,109.0,124.000000,268.000000,0.0,0.0,0.0,0.0,32.0,15.0,11.0,0.0,13.0,6.0,4.0,0.0,2.0,1.0,1.0,0.0,17.0,8.0,6.0,0.0
9,test,Bike Pink,6045,23521,6560,5718,5173,2719,17,3283,117.0,130.000000,277.000000,19806.0,2589.0,1816.0,120.0,2803.0,984.0,653.0,4.0,1517.0,526.0,323.0,1.0,541.0,207.0,132.0,0.0,746.0,251.0,198.0,3.0


In [27]:
df_analysis5 =  df_merge\
.groupby(['group_tc','city_name', 'service_name'])\
.agg(**telemetri_agg_dict)\
.sort_values(by=['city_name', 'service_name', 'group_tc'])\
.reset_index()

df_analysis5[df_analysis5['customers'] > 0]

,group_tc,city_name,service_name,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,Bangalore,Auto,458934,1757817,713101,13789,462908,261769,1471,301613,66.000000,130.666667,280.000000,0.0,0.0,0.0,0.0,45282.0,14810.0,9961.0,43.0,9268.0,3370.0,1919.0,5.0,7544.0,2973.0,1970.0,6.0,28505.0,8467.0,6072.0,32.0
1,test,Bangalore,Auto,423102,1696601,673693,18358,495158,233714,1172,271447,48.000000,132.500000,289.000000,2658020.0,366800.0,226585.0,3598.0,4665167.0,1892663.0,1036872.0,4362.0,2110387.0,912658.0,407003.0,1861.0,947039.0,423470.0,230440.0,356.0,1611259.0,556537.0,399430.0,2145.0
2,control,Bangalore,Auto Pool,16921,84368,16573,10824,25757,8445,96,22561,81.000000,136.000000,295.000000,0.0,0.0,0.0,0.0,13075.0,5915.0,3176.0,8.0,6450.0,3389.0,1514.0,3.0,2308.0,1046.0,622.0,0.0,4329.0,1480.0,1040.0,5.0
3,test,Bangalore,Auto Pool,6930,44817,7372,5345,17300,4071,28,10647,75.000000,137.750000,312.500000,206.0,33.0,20.0,2.0,7112.0,2925.0,1628.0,7.0,4155.0,1850.0,909.0,5.0,963.0,427.0,243.0,0.0,1998.0,648.0,476.0,2.0
4,control,Bangalore,Bike Lite,7431,26385,9336,197,10309,3446,32,2951,70.000000,131.000000,366.000000,0.0,0.0,0.0,0.0,933.0,288.0,229.0,3.0,258.0,90.0,60.0,1.0,147.0,58.0,46.0,1.0,528.0,140.0,123.0,1.0
5,test,Bangalore,Bike Lite,7247,28868,9949,449,11569,3672,34,3066,86.000000,134.000000,362.000000,30017.0,3255.0,2208.0,28.0,37782.0,13371.0,7882.0,34.0,18747.0,7054.0,3561.0,12.0,7117.0,2810.0,1726.0,3.0,11941.0,3507.0,2595.0,19.0
10,control,Bangalore,CabEconomy,193595,581562,219703,5092,169674,103131,648,82720,101.000000,140.000000,322.000000,0.0,0.0,0.0,0.0,530516.0,249691.0,130484.0,703.0,231354.0,129030.0,55668.0,347.0,99900.0,49380.0,28972.0,75.0,199624.0,71282.0,45845.0,281.0
11,test,Bangalore,CabEconomy,62711,314744,82060,4807,131929,45914,177,49516,78.000000,139.200000,314.000000,27993.0,4256.0,2380.0,69.0,316043.0,138602.0,69832.0,388.0,151996.0,74134.0,31162.0,243.0,57019.0,27799.0,15284.0,32.0,107220.0,36669.0,23386.0,113.0
12,control,Bangalore,Link,459146,1528776,675417,16205,429934,214341,2313,183693,87.000000,130.000000,342.000000,0.0,0.0,0.0,0.0,1861417.0,728950.0,406746.0,1888.0,896089.0,379557.0,179076.0,837.0,373591.0,154369.0,88133.0,119.0,593199.0,195025.0,139538.0,932.0
13,test,Bangalore,Link,99063,525207,159386,9194,212237,69738,434,72942,73.000000,128.750000,305.000000,50864.0,5967.0,3824.0,41.0,575567.0,221507.0,120699.0,574.0,298821.0,120341.0,56190.0,303.0,109291.0,47395.0,26021.0,32.0,167914.0,53772.0,38488.0,239.0


In [28]:
df_analysis4.to_clipboard(index=False)

In [29]:
df_analysis1

,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time,css_request,css_response,css_viewable_impression,css_click,pos_request,pos_response,pos_viewable_impression,pos_click,onTheWay_request,onTheWay_response,onTheWay_viewable_impression,onTheWay_click,arrived_request,arrived_response,arrived_viewable_impression,arrived_click,started_request,started_response,started_viewable_impression,started_click
0,control,3827634,13377738,7031955,159006,2638363,2220827,18794,1261563,97.0,130.0,330.0,0.0,0.0,0.0,0.0,14238008.0,5727658.0,3139853.0,14232.0,6541343.0,2994635.0,1410823.0,7226.0,3022588.0,1330373.0,793866.0,1610.0,4684086.0,1402658.0,935167.0,5396.0
1,test,1725329,8016759,3775785,130314,2009952,1262326,6647,810765,95.0,135.0,320.0,9463860.0,1215643.0,813401.0,11519.0,22459461.0,8760719.0,4821155.0,21799.0,10499270.0,4545500.0,2040262.0,10288.0,4543297.0,1977649.0,1140039.0,2042.0,7432483.0,2237577.0,1640857.0,9469.0


In [30]:
df_analysis1['g2n'] = calculate_percentage(df_analysis1, 'net_orders', 'gross_orders')
df_analysis1['% cobrm'] = calculate_percentage(df_analysis1, 'cobrm', 'gross_orders')
df_analysis1['% cobra'] = calculate_percentage(df_analysis1, 'cobra', 'gross_orders')
df_analysis1['% ocara'] = calculate_percentage(df_analysis1, 'ocara', 'gross_orders')

df_analysis1[[ #'city_name', 
#          'service_name', 
         'group_tc','customers', 'g2n', '% cobrm', '% cobra', '% ocara']]

,group_tc,customers,g2n,% cobrm,% cobra,% ocara
0,control,3827634,52.56,1.19,19.72,16.60
1,test,1725329,47.10,1.63,25.07,15.75


In [31]:
df_data.group_tc.value_counts(True)

group_tc
control    0.678591
test       0.321409
Name: proportion, dtype: float64